
# MỨC 4: RANDOM FOREST – K-FOLD CROSS-VALIDATION

## 1. Giới thiệu

**Đề tài:** Dự đoán giá kim cương bằng Random Forest.

**Dataset:** Diamonds Prices2022.csv.

**Phương pháp:** Nested Cross-validation.

**Mục tiêu:** Đánh giá Random Forest khi mở rộng phạm vi tìm kiếm siêu tham số.

Quy trình:
1. Đọc và tiền xử lý dữ liệu giống mức 2 và mức 3.
2. Chia dữ liệu thành 5 outer fold.
3. Trong mỗi outer fold, sử dụng GridSearchCV với 3-Fold trên Train.
4. Chọn cấu hình có RMSE CV thấp nhất.
5. Đánh giá trên outer Test.
6. Tính trung bình kết quả của 5 fold.

Không sử dụng outer Test để lựa chọn siêu tham số.


Đọc dữ liệu

In [1]:

# CELL 2: IMPORT THU VIEN

import numpy as np  # Tinh toan so hoc
import pandas as pd  # Xu ly du lieu

from pathlib import Path  # Xu ly duong dan

from sklearn.model_selection import KFold  # Chia du lieu K-Fold
from sklearn.model_selection import GridSearchCV  # Tim sieu tham so

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa du lieu
from sklearn.compose import ColumnTransformer  # Xu ly cac cot
from sklearn.pipeline import Pipeline  # Ket hop cac buoc

from sklearn.ensemble import RandomForestRegressor  # Random Forest

from sklearn.metrics import mean_squared_error  # MSE
from sklearn.metrics import mean_absolute_error  # MAE
from sklearn.metrics import r2_score  # R2

print("Da import thu vien thanh cong!")

# CELL 4: DOC DU LIEU

# Duong dan khi chay tu thu muc notebook
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Duong dan du phong khi chay tu thu muc goc
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Kiem tra tep co ton tai khong
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay Diamonds Prices2022.csv")

# Doc CSV va bo cot chi so ban dau
diamonds = pd.read_csv(file_path, index_col=0)

# In kich thuoc
print("Kich thuoc ban dau:", diamonds.shape)

# Dem gia tri thieu
print("Gia tri thieu:", diamonds.isnull().sum().sum())

# Dem dong trung lap
print("Dong trung lap:", diamonds.duplicated().sum())

# Hien thi 5 dong dau
display(diamonds.head())


Da import thu vien thanh cong!
Kich thuoc ban dau: (53943, 10)
Gia tri thieu: 0
Dong trung lap: 149


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [2]:

from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
if 'Unnamed: 0' in diamonds.columns:
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (x, y, z <= 0)
initial_rows = len(diamonds)

diamonds = diamonds[
    (diamonds['x'] > 0) &
    (diamonds['y'] > 0) &
    (diamonds['z'] > 0)
].copy()

invalid_dimensions = initial_rows - len(diamonds)

print(
    f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu "
    "(x, y, z <= 0)."
)

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
num_duplicates = diamonds.duplicated().sum()

diamonds = diamonds.drop_duplicates()

print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

clarity_categories = [
    'I1', 'SI2', 'SI1', 'VS2',
    'VS1', 'VVS2', 'VVS1', 'IF'
]

encoder = OrdinalEncoder(
    categories=[
        cut_categories,
        color_categories,
        clarity_categories
    ]
)

diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(
    diamonds[['cut', 'color', 'clarity']]
)

# 5: Kiểm tra dữ liệu sau tiền xử lý
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")
print(diamonds.head())



Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75



## 3. Tiền xử lý dữ liệu

Thực hiện giống mức 2, mức 3 và mức 4 Hold-out:

1. Loại bỏ giá trị thiếu.
2. Loại bỏ mẫu có x, y hoặc z nhỏ hơn hoặc bằng 0.
3. Xóa các dòng trùng lặp.
4. Đặt lại chỉ số dòng.

Không thêm đặc trưng mới.


In [2]:

# CELL 6: TIEN XU LY

# Sao chep du lieu
df = diamonds.copy()

# Luu so dong ban dau
initial_rows = len(df)

# Xoa dong co gia tri thieu
df = df.dropna()

# Luu so dong sau khi xoa gia tri thieu
after_missing = len(df)

# Loai kich thuoc khong hop le
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Luu so dong sau khi loc kich thuoc
after_dimensions = len(df)

# Xoa cac dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong
df = df.reset_index(drop=True)

# In so dong bi loai
print("Dong thieu bi loai:", initial_rows - after_missing)

# In so dong kich thuoc khong hop le
print("Dong kich thuoc khong hop le:",
      after_missing - after_dimensions)

# In so dong trung lap
print("Dong trung lap bi loai:",
      after_dimensions - len(df))

# In so dong con lai
print("So dong sau tien xu ly:", len(df))

# Hien thi du lieu
display(df.head())


Dong thieu bi loai: 0
Dong kich thuoc khong hop le: 20
Dong trung lap bi loai: 148
So dong sau tien xu ly: 53775


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75



## 4. Chuẩn bị dữ liệu và K-Fold

Tách dữ liệu:
- X: Các thuộc tính đầu vào.
- y: Giá kim cương.

Sử dụng:
- Outer K-Fold: 5 fold để đánh giá.
- Inner K-Fold: 3 fold để tìm kiếm siêu tham số.

Cả hai đều sử dụng shuffle=True và random_state=42.


In [3]:

# CELL 8: CHUAN BI K-FOLD

# Lay cac thuoc tinh dau vao
X = df.drop(columns=["price"])

# Lay gia kim cuong
y = df["price"]

# Chia 5 fold ben ngoai
outer_kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Chia 3 fold ben trong
inner_kf = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# In so mau
print("Tong so mau:", len(X))

# In so fold
print("Outer folds:", outer_kf.get_n_splits())
print("Inner folds:", inner_kf.get_n_splits())


Tong so mau: 53775
Outer folds: 5
Inner folds: 3



## 5. Xây dựng Pipeline

Sử dụng OrdinalEncoder để mã hóa:
- color
- clarity
- cut

Kết hợp bộ mã hóa và Random Forest trong Pipeline.

Các thuộc tính số được giữ nguyên.

Bộ mã hóa được fit riêng trên Train của từng fold.


In [4]:

# CELL 10: TAO PIPELINE

# Thu tu mau sac
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu do tinh khiet
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu giac cat
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Khoi tao OrdinalEncoder
encoder = OrdinalEncoder(
    categories=[
        color_order,
        clarity_order,
        cut_order
    ]
)

# Tao bo tien xu ly
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", encoder, categorical_cols)
    ],
    remainder="passthrough"
)

# Khoi tao Random Forest
rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Tao Pipeline
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

print("Da tao Pipeline thanh cong!")


Da tao Pipeline thanh cong!



## 6. Phạm vi tìm kiếm siêu tham số

Sử dụng cùng phạm vi với mức 4 Hold-out:

- n_estimators: [100, 150, 200]
- max_depth: [None, 20, 30]
- min_samples_leaf: [1, 2]
- max_features: [0.8, 1.0]

Tổng cộng 36 cấu hình.

Mỗi cấu hình được đánh giá bằng 3-Fold trên Train của outer fold.


In [5]:

# CELL 12: KHAI BAO PHAM VI TIM KIEM

# Danh sach cac gia tri can thu
param_grid = {
    "model__n_estimators": [100, 150, 200],
    "model__max_depth": [None, 20, 30],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": [0.8, 1.0]
}

# Tinh so cau hinh
number_of_configs = (
    len(param_grid["model__n_estimators"])
    * len(param_grid["model__max_depth"])
    * len(param_grid["model__min_samples_leaf"])
    * len(param_grid["model__max_features"])
)

# In so cau hinh
print("So cau hinh:", number_of_configs)

# In tong so luot fit trong CV
print("So luot CV:", number_of_configs * 3 * 5)


So cau hinh: 36
So luot CV: 540



## 7. Huấn luyện Nested Cross-validation

Với mỗi outer fold:
1. Chia Train và Test.
2. Tìm siêu tham số bằng GridSearchCV với 3-Fold trên Train.
3. Lấy mô hình tốt nhất.
4. Dự đoán trên outer Test.
5. Tính MSE, RMSE, MAE và R².
6. Lưu kết quả.

Lặp lại cho 5 outer fold.


In [6]:

# CELL 14: CHAY NESTED K-FOLD

# Tao danh sach luu ket qua
results = []

# Lap qua 5 outer fold
for fold, (train_idx, test_idx) in enumerate(
    outer_kf.split(X), start=1
):

    # Lay du lieu Train
    X_train = X.iloc[train_idx].copy()

    # Lay du lieu Test
    X_test = X.iloc[test_idx].copy()

    # Lay nhan Train
    y_train = y.iloc[train_idx]

    # Lay nhan Test
    y_test = y.iloc[test_idx]

    # Tao GridSearchCV cho fold hien tai
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=inner_kf,
        n_jobs=1,
        refit=True,
        verbose=0
    )

    # Tim sieu tham so tren Train
    grid_search.fit(X_train, y_train)

    # Lay mo hinh tot nhat
    best_model = grid_search.best_estimator_

    # Du doan tren outer Test
    y_pred = best_model.predict(X_test)

    # Tinh MSE
    mse = mean_squared_error(y_test, y_pred)

    # Tinh RMSE
    rmse = np.sqrt(mse)

    # Tinh MAE
    mae = mean_absolute_error(y_test, y_pred)

    # Tinh R2
    r2 = r2_score(y_test, y_pred)

    # Lay sieu tham so tot nhat
    best_params = grid_search.best_params_

    # Luu ket qua fold
    results.append({
        "Fold": fold,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "RMSE_CV": -grid_search.best_score_,
        "n_estimators": best_params["model__n_estimators"],
        "max_depth": best_params["model__max_depth"],
        "min_samples_leaf": best_params["model__min_samples_leaf"],
        "max_features": best_params["model__max_features"]
    })

    # In ket qua fold
    print(f"Fold {fold} hoan thanh")
    print(f"  RMSE Test: {rmse:.4f}")
    print(f"  RMSE CV: {-grid_search.best_score_:.4f}")
    print(f"  Best params: {best_params}")

# Chuyen ket qua thanh DataFrame
results_df = pd.DataFrame(results)

# Hien thi ket qua
display(results_df.round(4))


Fold 1 hoan thanh
  RMSE Test: 520.8105
  RMSE CV: 547.8502
  Best params: {'model__max_depth': 20, 'model__max_features': 0.8, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Fold 2 hoan thanh
  RMSE Test: 530.6417
  RMSE CV: 549.8270
  Best params: {'model__max_depth': None, 'model__max_features': 0.8, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Fold 3 hoan thanh
  RMSE Test: 528.9463
  RMSE CV: 548.3949
  Best params: {'model__max_depth': None, 'model__max_features': 0.8, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Fold 4 hoan thanh
  RMSE Test: 530.7279
  RMSE CV: 551.1107
  Best params: {'model__max_depth': 30, 'model__max_features': 0.8, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Fold 5 hoan thanh
  RMSE Test: 564.5507
  RMSE CV: 546.1037
  Best params: {'model__max_depth': 30, 'model__max_features': 0.8, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}


,Fold,MSE,RMSE,MAE,R2,RMSE_CV,n_estimators,max_depth,min_samples_leaf,max_features
0,1,271243.6029,520.8105,264.4375,0.9828,547.8502,200,20.0,1,0.8
1,2,281580.6429,530.6417,262.6467,0.9819,549.8270,200,NaN,2,0.8
2,3,279784.2022,528.9463,259.7699,0.9824,548.3949,200,NaN,2,0.8
3,4,281672.0956,530.7279,265.4016,0.9827,551.1107,200,30.0,2,0.8
4,5,318717.5103,564.5507,267.9306,0.9800,546.1037,200,30.0,2,0.8



## 8. Tổng hợp kết quả 5-Fold

Tính trung bình và độ lệch chuẩn của:
- MSE
- RMSE
- MAE
- R²

Các chỉ số được tính trên 5 outer Test.


In [7]:

# CELL 16: TONG HOP KET QUA

# Danh sach chi so
metrics = ["MSE", "RMSE", "MAE", "R2"]

# Tinh gia tri trung binh
mean_scores = results_df[metrics].mean()

# Tinh do lech chuan mau
std_scores = results_df[metrics].std()

# Tao bang tong hop
summary = pd.DataFrame({
    "Chi so": metrics,
    "Trung binh": mean_scores.values,
    "Do lech chuan": std_scores.values
})

# Hien thi bang
display(summary.round(4))

# In ket qua
print("MUC 4 - RANDOM FOREST NESTED 5-FOLD")
print("-" * 45)
print(f"MSE trung binh : {mean_scores['MSE']:.4f}")
print(f"RMSE trung binh: {mean_scores['RMSE']:.4f}")
print(f"RMSE do lech chuan: {std_scores['RMSE']:.4f}")
print(f"MAE trung binh : {mean_scores['MAE']:.4f}")
print(f"R2 trung binh  : {mean_scores['R2']:.4f}")


,Chi so,Trung binh,Do lech chuan
0,MSE,286599.6108,18461.3562
1,RMSE,535.1354,16.9439
2,MAE,264.0373,3.0546
3,R2,0.9820,0.0011


MUC 4 - RANDOM FOREST NESTED 5-FOLD
---------------------------------------------
MSE trung binh : 286599.6108
RMSE trung binh: 535.1354
RMSE do lech chuan: 16.9439
MAE trung binh : 264.0373
R2 trung binh  : 0.9820


In [8]:

# CELL 18: SO SANH KET QUA K-FOLD

# Ket qua muc 2
level2_rmse = 541.2329
level2_mae = 267.0808
level2_r2 = 0.9815

# Ket qua muc 3
level3_rmse = 538.1197
level3_mae = 265.0177
level3_r2 = 0.9818

# Tao bang so sanh
comparison = pd.DataFrame({
    "Muc": ["Muc 2", "Muc 3", "Muc 4"],
    "RMSE": [level2_rmse, level3_rmse, mean_scores["RMSE"]],
    "MAE": [level2_mae, level3_mae, mean_scores["MAE"]],
    "R2": [level2_r2, level3_r2, mean_scores["R2"]]
})

# Tinh chenh lech RMSE so voi muc 3
comparison["Chenh lech RMSE so voi Muc 3"] = (
    comparison["RMSE"] - level3_rmse
)

# Hien thi bang
display(comparison.round(4))

# Tinh phan tram thay doi RMSE
rmse_change = (
    (mean_scores["RMSE"] - level3_rmse)
    / level3_rmse * 100
)

# In ket qua
print(f"Thay doi RMSE: {rmse_change:+.2f}%")

# Nhan xet
if mean_scores["RMSE"] < level3_rmse:
    print("RMSE Muc 4 thap hon Muc 3.")

elif mean_scores["RMSE"] > level3_rmse:
    print("RMSE Muc 4 cao hon Muc 3.")

else:
    print("RMSE hai muc bang nhau.")


,Muc,RMSE,MAE,R2,Chenh lech RMSE so voi Muc 3
0,Muc 2,541.2329,267.0808,0.9815,3.1132
1,Muc 3,538.1197,265.0177,0.9818,0.0000
2,Muc 4,535.1354,264.0373,0.9820,-2.9843


Thay doi RMSE: -0.55%
RMSE Muc 4 thap hon Muc 3.
